
# EE 467 Final Project  
## Botnet Detection Using Machine Learning  

**Team Members:** Ethan Le, Dhruv Naik  

This notebook implements:
- Binary botnet detection (Botnet vs Non-Botnet)
- Scenario-based train/test splitting
- Leakage control
- Logistic Regression baseline
- Random Forest + XGBoost
- Autoencoder anomaly baseline
- Recall @ Fixed FPR evaluation



## 1. Imports and Configuration

This section loads required libraries and sets random seeds for reproducibility.


In [3]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from final_util import timeit, evaluate_model, recall_at_fixed_fpr, select_threshold_at_fpr

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)



## 2. Data Loading

We load CTU-13 NetFlow .binetflow files for selected scenarios.
They can be treated like CSV files
In this project we will train on scenerio 8, evaluating to find the best model.
Then we will cross-evaluate on scenerio 11, to ensure the model holds accross datasets. 

This is done by loading a .binetflow file into a DataFramea and cleaning the column names by stripping whitespace. Additionally it filters out rows labeled as 'Background' to ensure that only definitive 'Botnet' and 'Normal' traffic is retained for analysis. This is done to remove noise from the training dataset.  


In [4]:
# --- Section 2: Data Loading & CSV Conversion ---
DATA_DIR = "./"  # Update this to your local path

# Name: load_and_clean_binetflow
# Description: Loads a .binetflow file, cleans whitespace from column names, 
#              and filters out 'Background' traffic to ensure clean labels.
def load_and_clean_binetflow(filename):

    path = os.path.join(DATA_DIR, filename)
    print(f"Loading {filename}...")
    
    # binetflow files are CSVs, we strip whitespace from headers
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    
    # Filter for only definitive labels: 'Botnet' or 'Normal'
    # This removes 'Background' flows which are unlabeled and can introduce noise
    initial_count = len(df)
    df = df[df['Label'].str.contains('Botnet|Normal', case=False, na=False)].copy()
    
    print(f"Filtered {initial_count - len(df)} background rows. {len(df)} rows remaining.")
    return df

# Load Scenario 8 (Murlo) data
scenario_8_raw = load_and_clean_binetflow("capture20110816-3.binetflow")

Loading capture20110816-3.binetflow...
Filtered 2875281 background rows. 78949 rows remaining.



## 3. Label Processing (Binary Setup)

The Datafram has a 'Label' column that contains descriptive CTU-13 labels, instead we want a new 'binary_label' column. This function maps any label containing 'Botnet', 'C&C', or 'Malware' to 1 (indicating malicious activity), and all other labels to 0 (indicating normal traffic). 
This binary labeling is essential for training classification models that require a binary target variable.



In [5]:
# Name : process_labels
# Description: Processes the 'Label' column to create a binary label 'binary_label' 
#              where 1 indicates Botnet activity and 0 indicates Normal traffic.
def process_labels(df):
    df = df.copy()
    
    # CTU-13 labels often look like 'flow=From-Botnet-V1-TCP-HTTP-1'
    # We use string contains to capture all variations of malicious activity
    df['binary_label'] = df['Label'].apply(
        lambda x: 1 if any(s in str(x) for s in ['Botnet', 'C&C', 'Malware']) else 0
    )
    
    return df

# --- Verify Loading and Labeling ---

# Process the labels
processed_df = process_labels(scenario_8_raw)

# Display examples of both Normal (0) and Botnet (1) traffic
cols_to_show = ['StartTime', 'Dur', 'Proto', 'SrcAddr', 'DstAddr', 'Label', 'binary_label']

print("--- Normal Traffic Verification (Label 0) ---")
display(processed_df[processed_df['binary_label'] == 0][cols_to_show].head(3))

print("\n--- Botnet Traffic Verification (Label 1) ---")
display(processed_df[processed_df['binary_label'] == 1][cols_to_show].head(3))

--- Normal Traffic Verification (Label 0) ---


,StartTime,Dur,Proto,SrcAddr,DstAddr,Label,binary_label
166,2011/08/16 14:18:56.645840,0.000328,udp,147.32.84.170,147.32.80.9,flow=From-Normal-V49-Stribrek,0
167,2011/08/16 14:18:56.646714,0.000410,udp,147.32.84.170,147.32.80.9,flow=From-Normal-V49-Stribrek,0
168,2011/08/16 14:18:56.647854,0.017853,tcp,147.32.84.170,209.85.148.106,flow=From-Normal-V49-Stribrek,0



--- Botnet Traffic Verification (Label 1) ---


,StartTime,Dur,Proto,SrcAddr,DstAddr,Label,binary_label
923,2011/08/16 14:19:01.438016,1.000221,udp,147.32.84.165,147.32.80.9,flow=From-Botnet-V49-UDP-DNS,1
1103,2011/08/16 14:19:02.439762,500.002869,tcp,147.32.84.165,195.113.232.98,flow=From-Botnet-V49-TCP-Established-HTTP-Ad-40,1
8354,2011/08/16 14:20:00.257453,0.000218,udp,147.32.84.165,147.32.80.9,flow=From-Botnet-V49-UDP-DNS,1



## 4. Feature Engineering and Leakage Control

We drop columns that might give the model information to cheat, for example the model might train off the source and destination IPs, ports, or the original text label.

Then we take a DataFrame and performs feature engineering by creating new features based on existing ones. We would like rate-based features such as bytes per second and packets per second. The function  `feature_engineering()` handles the categorical 'Proto' column using one-hot encoding to obtain these features. 

Afterwards we want to drops leakage columns, and separates the target variable (y) from the feature matrix (X). `prepare_features()` returns the cleaned feature matrix and target vector for model training.

In [6]:
# Columns to drop to prevent data leakage
DROP_COLUMNS = [
    'Label',      # Original text label (we use binary_label now)
    'SrcAddr',    # Source IP
    'DstAddr',    # Destination IP
    'StartTime',  # Timestamp
    'State',      # Connection state (text)
    'Dir',        # Directionality (text)
    'Sport',      # Source Port (often too high-cardinality for basic models)
    'Dport',      # Destination Port 
    'sTos',       # Source Type of Service
    'dTos'        # Destination Type of Service
]

# Name: feature_engineering
# Description: Takes a DataFrame and creates rate-based features
# (bytes per second and packets per second) using one-hot encoding from the 'Proto' column.
def feature_engineering(df):
    df = df.copy()
    
    # Create rate-based features using CTU-13 column names
    # Adding a small epsilon (1e-6) prevents division by zero if duration is 0
    df['bytes_per_sec'] = df['TotBytes'] / (df['Dur'] + 1e-6)
    df['pkts_per_sec'] = df['TotPkts'] / (df['Dur'] + 1e-6)
    
    # Handle Categorical Text Data: The 'Proto' (Protocol) column
    # We use One-Hot Encoding to turn 'tcp', 'udp', 'icmp' into separate 1 or 0 columns
    if 'Proto' in df.columns:
        # Group rare protocols into an 'other' category to keep feature space small
        df['Proto'] = df['Proto'].apply(lambda x: x if x in ['tcp', 'udp', 'icmp'] else 'other')
        proto_dummies = pd.get_dummies(df['Proto'], prefix='proto', dtype=int)
        df = pd.concat([df, proto_dummies], axis=1)
        df = df.drop(columns=['Proto'])
        
    return df

# Name: prepare_features
# Description: This function takes a DataFrame, drops leakage columns, and separates the target variable (y) from the feature matrix (X). 
def prepare_features(df):
    """
    Strips leakage columns and separates the target variable (y) 
    from the feature matrix (X).
    """
    df_cleaned = df.drop(columns=DROP_COLUMNS, errors='ignore')
    
    X = df_cleaned.drop(columns=['binary_label'], errors='ignore')
    y = df_cleaned['binary_label']
    
    return X, y

# --- Execute Feature Engineering & Preparation ---

# Apply feature engineering
engineered_df = feature_engineering(processed_df)

# Prepare features
X, y = prepare_features(engineered_df)

# Verify the resulting feature matrix
print("--- Feature Matrix X (First 5 rows) ---")
display(X.head())

print("\n--- Target Vector y (First 5 rows) ---")
display(y.head())

print(f"\nFinal shape of X: {X.shape}")

--- Feature Matrix X (First 5 rows) ---


,Dur,TotPkts,TotBytes,SrcBytes,bytes_per_sec,pkts_per_sec,proto_icmp,proto_other,proto_tcp,proto_udp
166,0.000328,2,400,74,1.215805e+06,6079.027356,0,0,0,1
167,0.000410,2,400,74,9.732360e+05,4866.180049,0,0,0,1
168,0.017853,7,478,272,2.677271e+04,392.069004,0,0,1,0
197,0.016182,2,120,60,7.415189e+03,123.586480,0,0,1,0
198,0.016431,2,120,60,7.302824e+03,121.713729,0,0,1,0



--- Target Vector y (First 5 rows) ---


166    0
167    0
168    0
197    0
198    0
Name: binary_label, dtype: int64


Final shape of X: (78949, 10)



## 5. Scenario-Based Train/Validation/Test Split

We want to Split Scenario 8 into 80% Training data and 20% Validation data. We use `stratify=y` to ensure both sets keep the same 90% Normal / 10% Botnet ratio. Leter on we can test on a future dataset from Scenario 11 to evaluate cross-scenario generalization.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE # Uses the seed defined in Section 1
)

# Print the shapes to verify the split
print("--- Data Split Verification ---")
print(f"Total samples: {len(X)}")
print(f"Training features (X_train): {X_train.shape}")
print(f"Validation features (X_test): {X_test.shape}")

# Verify the stratification worked (ratios should match)
print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True).map('{:.2%}'.format))

print("\nValidation Target Distribution:")
print(y_test.value_counts(normalize=True).map('{:.2%}'.format))

# Note for later: 
# To get X_test and y_test, you will load a second scenario (Scenario 11)
# run it through feature_engineering() and prepare_features(), 
# and use that entire matrix as your unseen test set!

--- Data Split Verification ---
Total samples: 78949
Training features (X_train): (63159, 10)
Validation features (X_test): (15790, 10)

Training Target Distribution:
binary_label
0    92.24%
1     7.76%
Name: proportion, dtype: object

Validation Target Distribution:
binary_label
0    92.24%
1     7.76%
Name: proportion, dtype: object



## 6. Scaling and Class Imbalance

Due to data imbalances and values, we need to standard scale the data so that the model does not overfit to one metric. We must do this for both the training and test data. Next, we need to adjust the class weights so that our model does not overfit to the normal traffic. Since our splits are 90% normal/10% botnet, a "dumb" classifier would simply guess normal. Adjusting the class weights gives more value to the botnet traffic.


In [8]:
# Initialize and fit the scaler ONLY on the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation data using the SAME scaler
X_test_scaled = scaler.transform(X_test)

# Compute class weights to handle the 90/10 imbalance
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Create a dictionary to pass into Logistic Regression and Random Forest
class_weight_dict = dict(zip(classes, weights))

# Verification Printout
print("--- Scaling & Class Weights Verification ---")
# If scaling worked, the mean should be practically 0 and standard deviation 1
print(f"X_train_scaled Mean: {np.mean(X_train_scaled):.2f}")
print(f"X_train_scaled Std Dev: {np.std(X_train_scaled):.2f}")

print(f"\nComputed Class Weights:")
for cls, weight in class_weight_dict.items():
    label_name = "Botnet (1)" if cls == 1 else "Normal (0)"
    print(f"{label_name}: {weight:.4f}")

--- Scaling & Class Weights Verification ---


X_train_scaled Mean: -0.00
X_train_scaled Std Dev: 1.00

Computed Class Weights:
Normal (0): 0.5421
Botnet (1): 6.4422



## 7. Logistic Regression Baseline

Now we create and test the the logistic regression model on the training set. From this we can calculate and evaluate the 1% FPR metric and see how the model fairs against the testing set. We are using probabilities (predict_proba) instead of hard predicitions because we want to achieve a 1% FPR. These allow us to calculate the ROC-AUC, threshold at 1% FPR, and the recall at 1% FPR. These metric are:

ROC-AUC: overall ability to distinguish traffic. 0.5 = random, 1.0 = perfect.

Threshold at 1% FPR: at what probability threshold will 99% of normal traffic be classified as normal, and 1% be incorrectly classified. A threshold of 0.45 would mean 99% of normal traffic scored a probability under 0.45.

Recall at 1% FPR: at the 1% FPR threshold, what percentage of botnet traffic is correctly classified as botnet.


In [9]:
print("--- Training Logistic Regression Baseline ---")

# Initialize the model
# We pass in the class_weight_dict we calculated in Section 6
with timeit("Training logistic regression classifier"):
    log_model = LogisticRegression(
        class_weight=class_weight_dict,
        max_iter=1000,
        random_state=RANDOM_STATE 
    ).fit(X_train_scaled, y_train)

print("Training complete.")

# Predict probabilities on the validation set
val_probs = log_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate using custom metrics
# Find the specific probability threshold that limits False Positives to 1%
threshold_1_fpr = select_threshold_at_fpr(y_test, val_probs, target_fpr=0.01)

# Calculate metrics
val_roc_auc = roc_auc_score(y_test, val_probs)
val_recall_at_1_fpr = recall_at_fixed_fpr(y_test, val_probs, target_fpr=0.01)

# Print the results
print(f"\n--- Validation Results (Scenario 8) ---")
print(f"ROC-AUC Score: {val_roc_auc:.4f}")
print(f"Threshold for 1% FPR: {threshold_1_fpr:.4f}")
print(f"Recall @ 1% FPR: {val_recall_at_1_fpr:.2%}")

# Show full classification report using our strict threshold
val_preds_strict = (val_probs >= threshold_1_fpr).astype(int)
print("\nClassification Report (using strict 1% FPR threshold):")
print(classification_report(y_test, val_preds_strict, target_names=['Normal (0)', 'Botnet (1)']))

--- Training Logistic Regression Baseline ---
Training logistic regression classifier started...
Training logistic regression classifier completed. Elapsed time: 0.07s

Training complete.

--- Validation Results (Scenario 8) ---
ROC-AUC Score: 0.9470
Threshold for 1% FPR: 0.3784
Recall @ 1% FPR: 76.57%

Classification Report (using strict 1% FPR threshold):
              precision    recall  f1-score   support

  Normal (0)       0.98      0.99      0.99     14565
  Botnet (1)       0.90      0.77      0.83      1225

    accuracy                           0.98     15790
   macro avg       0.94      0.88      0.91     15790
weighted avg       0.97      0.98      0.97     15790




## 8. Tree-Based Models

We implement:
- Random Forest
- XGBoost (boosted trees)

These capture nonlinear feature interactions.


In [12]:
print("--- 9.1 Training Random Forest ---")
# 1. Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight=class_weight_dict, 
    random_state=RANDOM_STATE,
    n_jobs=-1 
)

rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# Calculate custom metrics
rf_threshold = select_threshold_at_fpr(y_test, rf_probs, target_fpr=0.01)
rf_auc = roc_auc_score(y_test, rf_probs)
rf_recall = recall_at_fixed_fpr(y_test, rf_probs, target_fpr=0.01)

print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"Random Forest Threshold for 1% FPR: {rf_threshold:.4f}")
print(f"Random Forest Recall @ 1% FPR: {rf_recall:.2%}\n")

# Generate Classification Report using the strict threshold
rf_preds_strict = (rf_probs >= rf_threshold).astype(int)
print("Classification Report (using strict 1% FPR threshold):")
print(classification_report(y_test, rf_preds_strict, target_names=['Normal (0)', 'Botnet (1)']))


print("\n")
print("--- 9.2 Training XGBoost ---")
# Calculate the ratio for XGBoost scale_pos_weight
neg_cases = sum(y_train == 0)
pos_cases = sum(y_train == 1)
scale_pos = neg_cases / pos_cases

# 2. Initialize XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Calculate custom metrics
xgb_threshold = select_threshold_at_fpr(y_test, xgb_probs, target_fpr=0.01)
xgb_auc = roc_auc_score(y_test, xgb_probs)
xgb_recall = recall_at_fixed_fpr(y_test, xgb_probs, target_fpr=0.01)

print(f"XGBoost ROC-AUC: {xgb_auc:.4f}")
print(f"XGBoost Threshold for 1% FPR: {xgb_threshold:.4f}")
print(f"XGBoost Recall @ 1% FPR: {xgb_recall:.2%}\n")

# Generate Classification Report using the strict threshold
xgb_preds_strict = (xgb_probs >= xgb_threshold).astype(int)
print("Classification Report (using strict 1% FPR threshold):")
print(classification_report(y_test, xgb_preds_strict, target_names=['Normal (0)', 'Botnet (1)']))

--- 9.1 Training Random Forest ---
Random Forest ROC-AUC: 0.9996
Random Forest Threshold for 1% FPR: 0.0200
Random Forest Recall @ 1% FPR: 99.84%

Classification Report (using strict 1% FPR threshold):
              precision    recall  f1-score   support

  Normal (0)       1.00      0.99      1.00     14565
  Botnet (1)       0.90      1.00      0.95      1225

    accuracy                           0.99     15790
   macro avg       0.95      0.99      0.97     15790
weighted avg       0.99      0.99      0.99     15790



--- 9.2 Training XGBoost ---
XGBoost ROC-AUC: 0.9999
XGBoost Threshold for 1% FPR: 0.0297
XGBoost Recall @ 1% FPR: 99.84%

Classification Report (using strict 1% FPR threshold):
              precision    recall  f1-score   support

  Normal (0)       1.00      0.99      0.99     14565
  Botnet (1)       0.90      1.00      0.94      1225

    accuracy                           0.99     15790
   macro avg       0.95      0.99      0.97     15790
weighted avg       


## 9. Autoencoder (Anomaly Detection Baseline)

We:
- Train ONLY on normal traffic
- Use reconstruction error as anomaly score
- Select threshold at fixed FPR


In [ ]:

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)



## 10. Cross Scenerio Evaluation

Now we will evaluate our models on data from Scenerio 11.
